In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import os
import glob

## 1. 选择并加载单个 .pt 文件

In [5]:
DATA_DIR = "../data/rank3/rank3"

# 修改这里选择要查看的 step 和 layer
STEP = 20
LAYER = 55

filepath = os.path.join(DATA_DIR, f"step{STEP}_layer{LAYER}.pt")
snap = torch.load(filepath, map_location="cpu", weights_only=True)

print(f"File: {filepath}")
print(f"Keys: {list(snap.keys())}")

# print values
for k, v in snap.items():
    print(f"{k}: {v}")

print()

File: ../data/rank3/rank3\step20_layer55.pt
Keys: ['cumulative_expert_recv', 'wait_recv_cost', 'step', 'layer_id', 'rank', 'num_tokens']
cumulative_expert_recv: tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0], dtype=torch.int32)
wait_recv_cost: tensor([365087, 371926, 352184, 363416, 366251, 347396, 372116, 363061])
step: 20
layer_id: 55
rank: 3
num_tokens: 2



In [ ]:
for k, v in snap.items():
    if isinstance(v, torch.Tensor):
        print(f"{k}:")
        print(f"  shape = {v.shape}")
        print(f"  dtype = {v.dtype}")
        print(f"  sum   = {v.sum().item()}")
        print(f"  min   = {v.min().item()},  max = {v.max().item()}")
        print(f"  values: {v.tolist()}")
    else:
        print(f"{k}: {v}")
    print()

## 2. 查看该 layer 在所有 step 上的累积变化

In [ ]:
# 找到该 layer 所有可用的 step
all_files = sorted(glob.glob(os.path.join(DATA_DIR, f"step*_layer{LAYER}.pt")))
print(f"Found {len(all_files)} files for layer {LAYER}")

steps = []
wait_cumul = []
expert_cumul = []
tokens_list = []

for f in all_files:
    d = torch.load(f, map_location="cpu", weights_only=True)
    steps.append(d["step"])
    wait_cumul.append(d["wait_recv_cost"].numpy().astype(np.int64))
    expert_cumul.append(d["cumulative_expert_recv"].numpy().astype(np.int64))
    tokens_list.append(d["num_tokens"])

# 按 step 排序
order = np.argsort(steps)
steps = [steps[i] for i in order]
wait_cumul = [wait_cumul[i] for i in order]
expert_cumul = [expert_cumul[i] for i in order]
tokens_list = [tokens_list[i] for i in order]

wait_mat = np.stack(wait_cumul)    # [num_steps, group_size]
expert_mat = np.stack(expert_cumul) # [num_steps, num_local_experts]

print(f"Steps: {steps}")
print(f"wait_mat shape: {wait_mat.shape}")
print(f"expert_mat shape: {expert_mat.shape}")

## 3. 累积 wait_recv_cost 随 step 变化

In [ ]:
group_size = wait_mat.shape[1]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 累积曲线
for p in range(group_size):
    axes[0].plot(steps, wait_mat[:, p], marker=".", markersize=4, label=f"Peer {p}")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Cumulative Wait Recv Cost")
axes[0].set_title(f"Layer {LAYER} — Cumulative Wait per Peer")
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# Delta (per-step increment)
wait_delta = np.diff(wait_mat, axis=0)  # [num_steps-1, group_size]
delta_steps = steps[1:]
for p in range(group_size):
    axes[1].plot(delta_steps, wait_delta[:, p], marker=".", markersize=4, label=f"Peer {p}")
axes[1].set_xlabel("Step")
axes[1].set_ylabel("Wait Delta (per step)")
axes[1].set_title(f"Layer {LAYER} — Per-Step Wait Delta per Peer")
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. 当前 step 的 wait_recv_cost 各 peer 分布

In [ ]:
wait = snap["wait_recv_cost"].numpy()

fig, ax = plt.subplots(figsize=(8, 4))
colors = plt.cm.Set2(np.linspace(0, 1, len(wait)))
bars = ax.bar(range(len(wait)), wait, color=colors, edgecolor="gray", linewidth=0.5)
mean_val = wait.mean()
ax.axhline(mean_val, color="red", linestyle="--", linewidth=1, label=f"Mean = {mean_val:.0f}")

for bar, val in zip(bars, wait):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
            f"{val:,}", ha="center", va="bottom", fontsize=9)

ax.set_xlabel("Peer Rank")
ax.set_ylabel("Cumul. Wait Recv Cost")
ax.set_title(f"Step {STEP}, Layer {LAYER} — Wait per Peer")
ax.set_xticks(range(len(wait)))
ax.legend()
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

print(f"Sum:     {wait.sum():,}")
print(f"Mean:    {mean_val:,.0f}")
print(f"Std:     {wait.std():,.0f}")
print(f"CoV:     {wait.std()/max(mean_val,1):.4f}")
print(f"Max/Min: {wait.max()/max(wait.min(),1):.4f}")
print(f"Slowest: Peer {np.argmax(wait)}")
print(f"Fastest: Peer {np.argmin(wait)}")

## 5. Expert Recv 分布（如果非零）

In [ ]:
expert_recv = snap["cumulative_expert_recv"].numpy()

if expert_recv.sum() == 0:
    print("cumulative_expert_recv 全为零，该次运行未记录 expert 分布数据。")
else:
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.bar(range(len(expert_recv)), expert_recv, color="mediumpurple", alpha=0.8)
    ax.set_xlabel("Local Expert ID")
    ax.set_ylabel("Cumul. Recv Count")
    ax.set_title(f"Step {STEP}, Layer {LAYER} — Expert Recv Distribution")
    ax.grid(True, alpha=0.3, axis="y")
    plt.tight_layout()
    plt.show()

    print(f"Total recv:    {expert_recv.sum():,}")
    print(f"Mean per exp:  {expert_recv.mean():,.1f}")
    print(f"CoV:           {expert_recv.std()/max(expert_recv.mean(),1):.4f}")
    print(f"Hottest:       Expert {np.argmax(expert_recv)} ({expert_recv.max():,})")
    print(f"Coldest:       Expert {np.argmin(expert_recv)} ({expert_recv.min():,})")

## 6. 对比同一 step 不同 layer

In [ ]:
# 加载同一 step 的所有 layer
layer_files = sorted(glob.glob(os.path.join(DATA_DIR, f"step{STEP}_layer*.pt")))
layer_ids = []
layer_waits = []

for f in layer_files:
    d = torch.load(f, map_location="cpu", weights_only=True)
    layer_ids.append(d["layer_id"])
    layer_waits.append(d["wait_recv_cost"].numpy().astype(np.int64))

order = np.argsort(layer_ids)
layer_ids = [layer_ids[i] for i in order]
layer_waits = [layer_waits[i] for i in order]
layer_wait_mat = np.stack(layer_waits)  # [num_layers, group_size]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Heatmap
im = axes[0].imshow(layer_wait_mat, aspect="auto", cmap="YlOrRd", interpolation="nearest")
axes[0].set_xlabel("Peer Rank")
axes[0].set_ylabel("Layer ID")
axes[0].set_title(f"Step {STEP} — Cumul. Wait per Layer × Peer")
axes[0].set_xticks(range(group_size))
axes[0].set_yticks(range(len(layer_ids)))
axes[0].set_yticklabels(layer_ids, fontsize=6)
fig.colorbar(im, ax=axes[0], shrink=0.8)

# Total per layer
axes[1].barh(range(len(layer_ids)), layer_wait_mat.sum(axis=1), color="steelblue", alpha=0.8)
axes[1].set_xlabel("Total Wait")
axes[1].set_ylabel("Layer ID")
axes[1].set_title(f"Step {STEP} — Total Wait per Layer")
axes[1].set_yticks(range(len(layer_ids)))
axes[1].set_yticklabels(layer_ids, fontsize=6)
axes[1].grid(True, alpha=0.3, axis="x")
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## 7. 自由探索：批量加载所有数据

In [ ]:
# 加载全部数据到 dict，方便后续自由探索
all_data = {}
for f in glob.glob(os.path.join(DATA_DIR, "step*_layer*.pt")):
    d = torch.load(f, map_location="cpu", weights_only=True)
    all_data[(d["step"], d["layer_id"])] = d

all_steps = sorted(set(k[0] for k in all_data))
all_layers = sorted(set(k[1] for k in all_data))
print(f"Loaded {len(all_data)} snapshots")
print(f"Steps:  {all_steps}")
print(f"Layers: {all_layers}")
print()
print("用法: all_data[(step, layer_id)] 获取任意 snapshot")
print("示例: all_data[(10, 30)]['wait_recv_cost']")